In [ ]:
import os
import pandas as pd
import SimpleITK as sitk
import logging
import numpy as np
from radiomics import featureextractor

# --- 0. Setup and Initialization ---
logger = logging.getLogger("radiomics")
logger.setLevel(logging.ERROR)

# Define directories (Update these paths before execution)
BASE_DIR = r"./dataimage"
OUTPUT_CSV = "radiomics_features_EV_standardized.csv"

settings = {
    'binWidth': 25,
    'resampledPixelSpacing': [1, 1, 1],
    'interpolator': sitk.sitkBSpline,
    'geometryTolerance': 1e-3,
    'correctMask': True,
    'enableCExtensions': True
}

extractor = featureextractor.RadiomicsFeatureExtractor(**settings)
for cls in ['shape', 'firstorder', 'glcm', 'glrlm', 'glszm', 'gldm', 'ngtdm']:
    extractor.enableFeatureClassByName(cls)

def extract_features(image_obj, mask_obj, label_id, prefix):
    try:
        extractor.settings['label'] = label_id
        feature_vector = extractor.execute(image_obj, mask_obj)
        return {f"{prefix}_{k}": v for k, v in feature_vector.items() if not k.startswith("diagnostics")}
    except Exception as e:
        print(f"  [Warning] Extraction failed for {prefix} (Label {label_id}): {e}")
        return {}

# --- 1. Main Extraction Loop ---
data_list = []
classes = ['bleeding', 'non_bleeding']

for class_label in classes:
    class_path = os.path.join(BASE_DIR, class_label)
    if not os.path.exists(class_path): continue
        
    patient_folders = [f for f in os.listdir(class_path) if os.path.isdir(os.path.join(class_path, f))]
    print(f"\nProcessing cohort: {class_label.upper()} (n={len(patient_folders)})")
    
    for idx, patient_id in enumerate(patient_folders):
        patient_dir = os.path.join(class_path, patient_id)
        vol_path = os.path.join(patient_dir, "volume.nii.gz")
        target_path = os.path.join(patient_dir, "target.nii.gz")
        varix_path = os.path.join(patient_dir, "varix.nii.gz")
        
        if not (os.path.exists(vol_path) and os.path.exists(target_path) and os.path.exists(varix_path)):
            continue
            
        print(f"Processing ID: {patient_id}...", end="\r")
        try:
            sitk_vol = sitk.ReadImage(vol_path)
            sitk_target = sitk.ReadImage(target_path)
            sitk_varix = sitk.ReadImage(varix_path)
            
            patient_features = {'PatientID': patient_id, 'Label': 1 if class_label == 'bleeding' else 0}
            patient_features.update(extract_features(sitk_vol, sitk_target, 1, "Spleen"))
            patient_features.update(extract_features(sitk_vol, sitk_target, 2, "Liver"))
            patient_features.update(extract_features(sitk_vol, sitk_target, 3, "Esophagus"))
            patient_features.update(extract_features(sitk_vol, sitk_varix, 1, "Varix"))
            
            data_list.append(patient_features)
        except Exception as e:
            pass

if data_list:
    df = pd.DataFrame(data_list)
    cols = ['PatientID', 'Label'] + [c for c in df.columns if c not in ['PatientID', 'Label']]
    df[cols].to_csv(OUTPUT_CSV, index=False)
    print(f"\nExtraction completed successfully. Saved to {OUTPUT_CSV}")

In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegressionCV, LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from sklearn.pipeline import Pipeline

# --- 0. Reproducibility & Data Loading ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

RADIOMICS_PATH = "radiomics_features_EV_standardized.csv"
CLINICAL_PATH = "clinical_data.xlsx" # Update path

print(">>> Step 1: Loading Internal Data...")
df_rad = pd.read_csv(RADIOMICS_PATH)
df_rad['PatientID'] = df_rad['PatientID'].astype(str)

try:
    xls = pd.read_excel(CLINICAL_PATH, sheet_name=None)
    df_clin = pd.concat([d for _, d in xls.items()], ignore_index=True)
    df_clin.columns = [c.strip().lower() for c in df_clin.columns]
    id_col = [c for c in df_clin.columns if 'id' in c][0]
    df_clin.rename(columns={id_col: 'PatientID'}, inplace=True)
    df_clin['PatientID'] = df_clin['PatientID'].astype(str)
    
    df_master = pd.merge(df_rad, df_clin, on='PatientID', how='inner')
    if 'pvt' not in df_master.columns: df_master['pvt'] = 0
except Exception as e:
    print(f"Warning: Clinical data load failed. Using radiomics only. Error: {e}")
    df_master = df_rad.copy()

y = df_master['Label'].values

# --- 1. Bootstrapping Metrics Definition ---
def compute_metrics_with_ci(y_true, y_probs, threshold, n_bootstraps=1000, seed=SEED):
    y_true, y_probs = np.array(y_true), np.array(y_probs)
    y_pred = (y_probs >= threshold).astype(int)
    
    auc_score = roc_auc_score(y_true, y_probs)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sens_score = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec_score = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    auc_boots, sens_boots, spec_boots = [], [], []
    rng = np.random.RandomState(seed)
    for _ in range(n_bootstraps):
        idx = rng.randint(0, len(y_probs), len(y_probs))
        if len(np.unique(y_true[idx])) < 2: continue
        try:
            auc_boots.append(roc_auc_score(y_true[idx], y_probs[idx]))
            tn_b, fp_b, fn_b, tp_b = confusion_matrix(y_true[idx], y_pred[idx]).ravel()
            sens_boots.append(tp_b / (tp_b + fn_b) if (tp_b + fn_b) > 0 else 0)
            spec_boots.append(tn_b / (tn_b + fp_b) if (tn_b + fp_b) > 0 else 0)
        except: continue

    def get_ci(scores):
        if not scores: return 0, 0
        s = np.sort(scores)
        return s[int(0.025 * len(s))], s[int(0.975 * len(s))]

    return {
        "AUC": (auc_score, *get_ci(auc_boots)),
        "Sens": (sens_score, *get_ci(sens_boots)),
        "Spec": (spec_score, *get_ci(spec_boots))
    }

# --- 2. Model Training Pipeline (Model E: Varix + Clinical) ---
print(">>> Step 2: Training Integrated Model (LASSO)...")
varix_cols = [c for c in df_master.columns if str(c).startswith('Varix')]
target_features = varix_cols + ['pvt']

X_full = df_master[target_features]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scenario_probs = np.zeros(len(y))

# Assuming robust features derived from internal nested CV (e.g., SVR, Coarseness)
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('clf', LogisticRegressionCV(Cs=10, cv=5, penalty='l1', solver='liblinear', class_weight='balanced', random_state=SEED))
])

# Generate Cross-Validated Probabilities
for train_idx, val_idx in cv.split(X_full, y):
    X_train, X_val = X_full.iloc[train_idx], X_full.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    pipeline.fit(X_train, y_train)
    scenario_probs[val_idx] = pipeline.predict_proba(X_val)[:, 1]

# Evaluate
from sklearn.metrics import roc_curve
fpr, tpr, thres = roc_curve(y, scenario_probs)
best_th = thres[np.argmax(tpr - fpr)]

metrics = compute_metrics_with_ci(y, scenario_probs, best_th)
print(f"Internal Validation Completed.")
print(f"AUC: {metrics['AUC'][0]:.3f} (95% CI: {metrics['AUC'][1]:.3f}-{metrics['AUC'][2]:.3f})")
print(f"Sensitivity: {metrics['Sens'][0]:.3f} | Specificity: {metrics['Spec'][0]:.3f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import brier_score_loss
import shap
from sklearn.base import clone

print(">>> Step 3: External Test Set Evaluation...")

# [Placeholder] Load your external dataset here (X_test, y_test)
# Example: 
# df_ext = pd.read_csv("External_Features_Final.csv")
# X_test = df_ext[target_features]
# y_test = df_ext['Label'].values

# Assuming X_test and y_test are defined, fit the final model on the entire internal dataset
final_pipeline = clone(pipeline)
final_pipeline.fit(X_full, y)

# Predict on External Test Set
# ext_probs = final_pipeline.predict_proba(X_test)[:, 1]
# ext_metrics = compute_metrics_with_ci(y_test, ext_probs, best_th)
# ext_brier = brier_score_loss(y_test, ext_probs)

# print(f"External Test Set AUC: {ext_metrics['AUC'][0]:.3f}")
# print(f"Brier Score: {ext_brier:.4f}")

# --- SHAP Analysis ---
print(">>> Step 4: Generating SHAP Explanations...")
clf = final_pipeline.named_steps['clf']
scaler = final_pipeline.named_steps['scaler']
imputer = final_pipeline.named_steps['imputer']

X_full_processed = pd.DataFrame(scaler.transform(imputer.transform(X_full)), columns=X_full.columns)
explainer = shap.LinearExplainer(clf, X_full_processed)
shap_values = explainer.shap_values(X_full_processed)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_full_processed, show=False)
plt.title("SHAP Summary: Integrated Radiomics-Clinical Model")
plt.tight_layout()
plt.savefig("SHAP_Summary.png", dpi=300)
plt.show()

print("Execution Complete. Results and figures saved.")